In [ ]:

%%capture
import import_ipynb

from separability import *


In [ ]:

def mismatched(W, X, y):
    'Boolean array: which samples have y_i <= y_j for some j != i.'
    Y = X @ W.T
    rows = np.arange(len(y))
    own = Y[rows, y]
    Y[rows, y] = -np.inf
    return own <= Y.max(1)

def perceptron(X, y, t=1.0, max_updates=200_000):
    'Return (W, number of updates, number of sweeps, converged).'
    d = y.max() + 1
    W = np.zeros((d, X.shape[1]))
    n = sweeps = 0
    while n < max_updates:
        sweeps += 1
        clean = True
        for k in range(len(X)):
            yk = W @ X[k]
            i = y[k]
            z = yk.copy(); z[i] = -np.inf
            j = int(np.argmax(z))
            if yk[i] <= yk[j]:	# x_k is mismatched
                W[i] += t*X[k]
                W[j] -= t*X[k]
                n += 1
                clean = False
        if clean:
            return W, n, sweeps, True
    return W, n, sweeps, False

def gd_first_separation(X, y, iters=200_000):
    'The gradient descent of pinwheel_gd.ipynb, stopped at the first step with no mismatched sample.'
    Pm = np.eye(3)[y]
    t = 1/np.mean(np.sum(X**2, 1))
    W = np.zeros((3, 3))
    for n in range(1, iters + 1):
        W -= t*(softmax(X @ W.T, axis=1) - Pm).T @ X/len(X)
        if not mismatched(W, X, y).any():
            return W, n
    return W, None

def margin(W, X, y):
    'Smallest gap y_i - y_j of the unit weight W/|W|.'
    Y = X @ (W/np.linalg.norm(W)).T
    rows = np.arange(len(y))
    own = Y[rows, y]
    Y[rows, y] = -np.inf
    return (own - Y.max(1)).min()


In [ ]:

perc = {}
for (a, b), r in runs.items():
    if not r['separable']:
        continue
    X, y = r['X'], r['y']
    N = len(X)
    t0 = perf_counter(); Wp, updates, sweeps, converged = perceptron(X, y); sec_p = perf_counter() - t0
    t0 = perf_counter(); Wg1, first = gd_first_separation(X, y); sec_g1 = perf_counter() - t0
    steps_g = int(r['norms'][-1, 0])
    perc[(a, b)] = dict(W=Wp, updates=updates, W_gd=Wg1, n_gd=first)
    print(f'(a,b) = ({a}, {b})')
    print(f'   {'':34s} {'steps':>9s} {'sample evals':>14s} {'seconds':>9s} {'margin':>10s}')
    print(f'   {'perceptron, to termination':34s} {updates:>9d} {sweeps*N:>14d} {sec_p:>9.3f} {margin(Wp, X, y):>10.1e}')
    print(f'   {'gradient descent, first separation':34s} {first:>9d} {first*N:>14d} {sec_g1:>9.3f} {margin(Wg1, X, y):>10.1e}')
    print(f'   {'gradient descent, n = 2e5':34s} {steps_g:>9d} {steps_g*N:>14d} {r['seconds']:>9.3f} {margin(r['W'], X, y):>10.1e}')
    print(f'   to first separation, gradient descent / perceptron: {first/updates:.3g} x the steps, '
          f'{first/sweeps:.3g} x the sample evaluations')
    r2 = np.max(np.sum(X[:, :2]**2, 1)); gam = margin(r['W'], X, y)
    print(f'   convergence theorem: n <= 2(1+r^2)/gamma^2 = {2*(1 + r2)/gam**2:.0f}, '
          f'with r^2 = {r2:.2f}, gamma = {gam:.4f} from the gradient descent weight')
    print()

X3, y3 = runs[(-2.5, 1.3)]['X'], runs[(-2.5, 1.3)]['y']
Wq, uq, sq, cq = perceptron(X3, y3, max_updates=20_000)
print(f'(a,b) = (-2.5, 1.3): perceptron converged = {cq} after {uq} updates ({sq} sweeps); '
      f'{int(mismatched(Wq, X3, y3).sum())} of {len(X3)} samples still mismatched')


In [ ]:

KEY = (-2.5, 1.1)
r, p = runs[KEY], perc[KEY]
X, y = r['X'], r['y']

fig, axs = plt.subplots(1, 3, figsize=(6.9, 2.1), gridspec_kw={'width_ratios': [1, 1, 1.25]})
regions(axs[0], r['C'], p['W'], f'(a) perceptron, {p['updates']} updates')
regions(axs[1], r['C'], p['W_gd'], f'(b) gradient descent, {p['n_gd']} steps')

ax = axs[2]
for key, ls, lab, yoff in [((-2.5, 1.1), '-', 'separable', 4), ((-2.5, 1.3), '--', 'not separable', -20)]:
    n = runs[key]['norms']
    ax.plot(n[:, 0], n[:, 1], ls=ls, c='k', lw=1.1)
    ax.annotate(lab, (n[-1, 0], n[-1, 1]), xytext=(-3, yoff), textcoords='offset points',
                ha='right', fontsize=7.5)
ax.set_xscale('log')
ax.set_xlabel('gradient descent step $n$'); ax.set_ylabel(r'$\|W_n\|$')
ax.set_title('(c) weight norm', fontsize=8)
ax.grid(True, which='major', lw=0.3, color='#cccccc'); ax.set_axisbelow(True)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
ax.set_ylim(0, 60); ax.set_xlim(1, 2e5)

plt.tight_layout(pad=0.3, w_pad=0.8)
plt.savefig('pinwheel_percep.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()
print(f'margins: perceptron {margin(p['W'], X, y):.2e}, gradient descent at first separation {margin(p['W_gd'], X, y):.2e}')
